In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)

df = pd.read_csv("../data/processed/processed.csv")
df.head()

,total_requests_above_p95,had_error,had_suspicious_ua,had_referrer_type_interno,had_referrer_type_api,had_referrer_type_nulo,had_bot_sessions,had_unknown_country,had_suspicious_post_usage,had_suspicious_high_freq_path,byte_size_group_menor_1kb,byte_size_group_mayor_8kb,night_activity_rate
0,False,False,False,False,False,False,False,False,False,False,False,True,0.0
1,False,False,False,False,False,False,False,False,False,False,False,False,0.0
2,False,False,False,False,False,False,False,False,True,True,False,False,0.0
3,False,False,False,False,False,False,False,False,False,True,False,False,0.0
4,False,False,False,False,False,False,False,False,False,False,False,False,0.0


In [ ]:


X_scaled = StandardScaler().fit_transform(df)
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

In [ ]:
from sklearn.cluster import KMeans

inertias = []
K_range = range(1, 20)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.plot(K_range, inertias, marker='o')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo')
plt.grid(True)
plt.show()


In [ ]:

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Agrega etiquetas al DataFrame
df['cluster'] = clusters


In [ ]:
plt.scatter(pca_result[:, 0], pca_result[:, 1], c=clusters, cmap='tab10', alpha=0.6)

# Calcular centroides para ubicar líneas y etiquetas
for i in np.unique(clusters):
    idx = np.where(clusters == i)
    x_mean = pca_result[idx, 0].mean()
    y_mean = pca_result[idx, 1].mean()
    
    # Coordenada de etiqueta fuera del gráfico (eje derecho)
    x_label = pca_result[:,0].max() + 1.0
    
    # Línea del centroide hacia la etiqueta
    plt.plot([x_mean, x_label], [y_mean, y_mean], linestyle='--', color='gray', linewidth=0.8)
    
    # Texto de la etiqueta
    plt.text(x_label, y_mean, f'Cluster {i}', fontsize=11, va='center', ha='left',
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

# Ajustar límites del gráfico para que se vea todo
plt.xlim(pca_result[:,0].min(), pca_result[:,0].max() + 2)

plt.title("IP Behavior Clusters (KMeans, PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('cluster').mean()
